In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS shopflow.gold")
spark.sql("SHOW SCHEMAS IN shopflow").show()
print("gold schema ready")

In [0]:
%sql
CREATE OR REPLACE TABLE shopflow.gold.daily_revenue AS
SELECT
  order_date,
  COUNT(order_id)              AS total_orders,
  SUM(revnew)                 AS total_revenue,
  ROUND(AVG(revnew), 2)       AS avg_order_value,
  SUM(quantity)                AS units_sold,
  COUNT(DISTINCT customer_id)  AS unique_customers
FROM shopflow.silver.orders
WHERE status = 'completed'
GROUP BY order_date
ORDER BY order_date;

In [0]:
%sql
CREATE OR REPLACE TABLE shopflow.gold.monthly_revenue AS
WITH monthly AS(
    SELECT
    order_month,
    COUNT(order_id)             AS total_orders,
    ROUND(SUM(revnew), 2)      AS total_revenue,
    ROUND(AVG(revnew), 2)      AS avg_order_value,
    COUNT(DISTINCT customer_id) AS unique_customers
  FROM shopflow.silver.orders
  WHERE status = 'completed'
  GROUP BY order_month
)
SELECT
order_month,
total_orders,
total_revenue,
avg_order_value,
unique_customers,
-- Month-over-month revenue growth %
ROUND(
    (total_revenue - LAG(total_revenue) OVER (ORDER BY order_month))/LAG(total_revenue) OVER (ORDER BY order_month)*100, 2) AS mmo_growth_pct
FROM monthly
ORDER BY order_month

In [0]:
%sql
CREATE OR REPLACE TABLE shopflow.gold.top_products AS
SELECT
  o.product_id,
  p.product_name,
  p.category,
  p.base_price,
  SUM(o.quantity)              AS units_sold,
  ROUND(SUM(o.revnew), 2)     AS total_revenue,
  COUNT(o.order_id)            AS total_orders,
  ROUND(AVG(o.revnew), 2)     AS avg_order_value,
  -- Rank by revenue
  RANK() OVER (ORDER BY SUM(o.revnew) DESC) AS revenue_rank
FROM shopflow.silver.orders o
JOIN shopflow.silver.product p
  ON o.product_id = p.product_id
WHERE o.status = 'completed'
GROUP BY o.product_id, p.product_name, p.category, p.base_price
ORDER BY total_revenue DESC;

In [0]:
%sql
DROP TABLE shopflow.gold.daily_reveneu;

In [0]:
%sql
DROP TABLE shopflow.gold.monthly_revenuea;
DROP TABLE shopflow.gold.monthly_reveneu;

In [0]:
%sql
CREATE OR REPLACE TABLE shopflow.gold.category_performance AS
SELECT
  p.category,
  COUNT(DISTINCT o.product_id)  AS products_sold,
  SUM(o.quantity)               AS units_sold,
  ROUND(SUM(o.revnew), 2)      AS total_revenue,
  ROUND(AVG(o.revnew), 2)      AS avg_order_value,
  -- Revenue share % across all categories
  ROUND(
    SUM(o.revnew) * 100.0
    / SUM(SUM(o.revnew)) OVER ()
  , 2) AS revenue_share_pct
FROM shopflow.silver.orders o
JOIN shopflow.silver.product p
  ON o.product_id = p.product_id
WHERE o.status = 'completed'
GROUP BY p.category
ORDER BY total_revenue DESC;

In [0]:
%sql
SELECT * FROM shopflow.gold.top_products;

In [0]:
%sql
CREATE OR REPLACE TABLE shopflow.gold.customer_segments AS
WITH customer_spend AS (
  SELECT
    o.customer_id,
    c.name,
    c.city,
    c.country,
    c.signup_date,
    COUNT(o.order_id)           AS total_orders,
    ROUND(SUM(o.revnew), 2)    AS total_spend,
    ROUND(AVG(o.revnew), 2)    AS avg_order_value,
    MIN(o.order_date)           AS first_order_date,
    MAX(o.order_date)           AS last_order_date
  FROM shopflow.silver.orders o
  JOIN shopflow.silver.customers c
    ON o.customer_id = c.customer_id
  WHERE o.status = 'completed'
  GROUP BY o.customer_id, c.name, c.city, c.country, c.signup_date
)
SELECT
  *,
  CASE
    WHEN total_spend >= 500  THEN 'High Value'
    WHEN total_spend >= 200  THEN 'Mid Value'
    ELSE                          'Low Value'
  END AS customer_segment
FROM customer_spend
ORDER BY total_spend DESC;

In [0]:
%sql
SELECT * FROM shopflow.gold.customer_segments;

In [0]:
%sql
-- Row counts for all Gold tables
SELECT 'daily_revenue'       AS tbl, COUNT(*) AS rows FROM shopflow.gold.daily_revenue
UNION ALL
SELECT 'monthly_revenue',           COUNT(*)           FROM shopflow.gold.monthly_revenue
UNION ALL
SELECT 'top_products',              COUNT(*)           FROM shopflow.gold.top_products
UNION ALL
SELECT 'category_performance',      COUNT(*)           FROM shopflow.gold.category_performance
UNION ALL
SELECT 'customer_segments',         COUNT(*)           FROM shopflow.gold.customer_segments;

In [0]:
%sql
-- Total revenue should match across all Gold tables
SELECT
  (SELECT ROUND(SUM(total_revenue),2) FROM shopflow.gold.daily_revenue)
    AS daily_total,
  (SELECT ROUND(SUM(total_revenue),2) FROM shopflow.gold.monthly_revenue)
    AS monthly_total,
  (SELECT ROUND(SUM(total_revenue),2) FROM shopflow.gold.category_performance)
    AS category_total;
-- All 3 numbers must be identical ✓